# 03 — 분석 환경 점검 (도구를 실제로 돌려 확인)

> 본문: [`03_setup.md`](03_setup.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 3초** (실측)

**이 노트북은 도구를 직접 돌립니다.** import 여부만 보는 게 아니라 **ANARCI 를 실제로 한 번 돌려** numbering 이 나오는지 확인해요.
각 절은 **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서예요.
내가 만든 결과는 `my_run/` 에 쌓이고, 저장소에 커밋된 `data/` 는 **대조군(레퍼런스)** 으로만 씁니다.
어느 단계를 건너뛰거나 실패해도 `resolve()` 가 `data/` 로 폴백해서 뒤 절이 계속 돌아가요.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

**Colab**: 이 셀을 그대로 실행하면 클론 → 챕터 폴더 이동 → 필요한 도구 설치까지 한 번에 끝나요.
**로컬**: 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "03_setup"
PIP_PKGS = "pandas anarci abnumber"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True        # ANARCI 계열은 HMMER 의 hmmscan 실행파일이 필요해요 (pip 로는 안 깔려요)
PIN_TRANSFORMERS = None    # IgFold 체크포인트가 요구하는 transformers 버전(없으면 None)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

if NEED_HMMER and shutil.which("hmmscan") is None:
    # ANARCI 는 내부적으로 hmmscan 을 호출해요. pip install anarci 만으로는 안 깔려요.
    if IN_COLAB:
        _run("apt-get -qq update")                       # 인덱스가 낡으면 install 이 404 로 죽는다
        _run("apt-get -qq install -y hmmer")             # ← ANARCI 가 부르는 hmmscan
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if "igfold" in PIP_PKGS and importlib.util.find_spec("pkg_resources") is None:
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 해요.
    _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
    importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


if PIN_TRANSFORMERS:
    # IgFold 체크포인트에는 옛 transformers 의 토크나이저 객체가 pickle 돼 있어요.
    # 최신 transformers(5.x) 로는 unpickle 이 실패해서(Trie/BasicTokenizer 없음) 버전을 맞춥니다.
    _ver = subprocess.run([sys.executable, "-c",
                           "import transformers;print(transformers.__version__)"],
                          capture_output=True, text=True).stdout.strip()
    if not _ver.startswith("4."):
        print(f"[transformers {_ver or 'none'} → {PIN_TRANSFORMERS}] IgFold 체크포인트 호환 버전으로 맞춥니다")
        _run(f'"{sys.executable}" -m pip -q install "transformers=={PIN_TRANSFORMERS}"')

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — 스택 점검 + ANARCI 스모크 테스트 (본문 3.5)

"설치됐다"의 진짜 기준은 **도구가 결과를 내놓는가**예요. 아래 셀은 라이브러리 import 를 확인하고,
곧바로 데모 항체를 **실제로 numbering** 해서 결과를 `my_run/setup_report.csv` 에 씁니다.

In [ ]:
import importlib.util, shutil, sys, time
import pandas as pd

rows = []
def check(kind, name, ok, detail=""):
    rows.append({"kind": kind, "item": name, "ok": "O" if ok else "X", "detail": detail})
    print(("O " if ok else "X "), f"{name:12s}", detail)

print("[현재 커널 python]", sys.executable)
for m, label in [("Bio","biopython"), ("pandas","pandas"), ("matplotlib","matplotlib"),
                 ("anarci","anarci"), ("abnumber","abnumber")]:
    check("python", label, importlib.util.find_spec(m) is not None)
for tool in ["hmmscan", "ANARCI"]:
    check("cli", tool, shutil.which(tool) is not None, shutil.which(tool) or "PATH에 없음")

# --- 스모크 테스트: 데모 항체를 실제로 numbering ---
try:
    from abnumber import Chain
    seqs, name = {}, None
    for line in open("data/demo_mab.fa"):
        line = line.strip()
        if line.startswith(">"): name = line[1:].split()[0]; seqs[name] = ""
        elif name: seqs[name] += line
    for sid, seq in seqs.items():
        t0 = time.time()
        ch = Chain(seq, scheme="imgt")
        rows.append({"kind": "smoke", "item": sid, "ok": "O",
                     "detail": f"chain_type={ch.chain_type} cdr3={ch.cdr3_seq} ({time.time()-t0:.2f}s)"})
        print("O  smoke       ", sid, "→ chain_type", ch.chain_type, "| CDR3", ch.cdr3_seq)
except Exception as e:
    rows.append({"kind": "smoke", "item": "abnumber/ANARCI", "ok": "X", "detail": f"{type(e).__name__}: {e}"})
    print("X  smoke        실패:", type(e).__name__, e)
    print("   → hmmscan 이 없으면 여기서 FileNotFoundError 가 나요 (본문 3.1 참고)")

report = pd.DataFrame(rows)
report.to_csv("my_run/setup_report.csv", index=False)
print("\nWrote: my_run/setup_report.csv")

## 2) 내 결과 확인 — 점검표

In [ ]:
import pandas as pd
rep = pd.read_csv(resolve("setup_report.csv"))
display(rep)
missing = rep[rep.ok == "X"]
print("문제 없음 " if missing.empty else "다음 항목을 먼저 해결하세요:")
for _, r in missing.iterrows():
    print("  -", r["item"], r["detail"])

## 3) 레퍼런스 대조 — numbering 결과가 정답과 같은가

`data/setup_expected.csv` 에는 같은 데모 항체를 ANARCI(IMGT)로 돌렸을 때 **나와야 하는 값**이 들어 있어요.
스모크 테스트 결과가 이것과 같으면 환경이 제대로 선 거예요.

In [ ]:
import pandas as pd
exp = pd.read_csv("data/setup_expected.csv")
display(exp)
rep = pd.read_csv("my_run/setup_report.csv")
smoke = rep[rep.kind == "smoke"].set_index("item")["detail"].to_dict()
for _, r in exp.iterrows():
    got = smoke.get(r["id"], "")
    ok = (f"chain_type={r['chain_type']}" in got) and (f"cdr3={r['cdr3_imgt']}" in got)
    print(("일치  " if ok else "불일치"), r["id"], "| 기대:", r["chain_type"], r["cdr3_imgt"], "| 내 결과:", got or "(없음)")

> 다음 → 본문 [04. numbering & germline](../04_numbering/04_numbering.md)